# Running DPF Simulations on Metal (Apple Silicon)

This notebook demonstrates how to use the **Metal GPU backend** for high-performance MHD simulations on macOS devices with Apple Silicon (M1/M2/M3).

The Metal backend uses `PyTorch` (MPS device) and custom Metal kernels for significant speedups over the Python/Numba backend while maintaining production-grade accuracy (conservative finite volumes).

In [ ]:
import logging
import matplotlib.pyplot as plt
import numpy as np
from dpf.config import SimulationConfig
from dpf.engine import SimulationEngine
from dpf.presets import get_preset

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("dpf")

## Check for Metal Availability

We can check if the Metal backend is available on this system.

In [ ]:
try:
    from dpf.metal.metal_solver import MetalMHDSolver
    if MetalMHDSolver.is_available():
        print("✅ Metal backend is available!")
    else:
        print("⚠️ Metal backend is NOT available (requires PyTorch MSP).")
except ImportError:
    print("❌ Could not import MetalMHDSolver.")

## Setup Simulation

We'll use the `tutorial` preset but override the backend to `metal`.

In [ ]:
config_dict = get_preset("tutorial")
config_dict["fluid"]["backend"] = "metal"
config = SimulationConfig(**config_dict)

print(f"Simulation backend: {config.fluid.backend}")
print(f"Grid shape: {config.grid_shape}")

## Run Simulation

We run the simulation for a short duration.

In [ ]:
engine = SimulationEngine(config)
summary = engine.run(max_steps=50)

## Visualize Results

Let's plot a cross-section of the plasma density.

In [ ]:
state = engine.state
rho = state["rho"]

if rho.ndim == 3:
    mid_z = rho.shape[2] // 2
    plt.figure(figsize=(10, 8))
    plt.imshow(rho[:, :, mid_z].T, origin="lower", cmap="plasma")
    plt.colorbar(label="Density (kg/m^3)")
    plt.title(f"Density Cross-section (z={mid_z})")
    plt.show()
else:
    print("Output is 2D, plotting directly.")
    plt.imshow(rho.T, origin="lower", cmap="plasma")
    plt.show()